# Assignment 28: Q&A RAG Chatbot with Message History

**Student:** Abhishek Thakare

This one takes the RAG pattern from my earlier assignments and adds actual
conversation memory on top of it - so the bot can handle "what about the
previous point?" instead of treating every question as if it just met me.

Reusing my onboarding knowledge base again (`notes.txt` + a new `policies.txt`
with more detail on the same policies, mainly so there's enough text across
the two files to actually need chunking).

**Same honest note as Assignment 25:** still no working OpenAI credits, and
this assignment's restriction is "LangChain + any LLM", so I'm sticking with
**Ollama running `llama3.2`** locally and the same Hugging Face embedding
model for the vector store.


## Before running this

- Ollama running locally with `llama3.2` pulled.
- `data/notes.txt` and `data/policies.txt` in a `data/` folder next to this
  notebook.
- If either the embedding model or Ollama isn't reachable, the retriever/chain
  cells print a clear message and set things to `None` instead of crashing -
  same pattern as my last couple of notebooks.


In [1]:
# Run this only if something is missing in your environment
# %pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-ollama faiss-cpu sentence-transformers

## PART 1 — Document Ingestion for RAG

### Task 1: Load Documents

Loading both text files this time instead of just one, using
`DirectoryLoader` so I don't have to load each file by hand.


In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data", glob="*.txt", loader_cls=TextLoader)
documents = loader.load()

print("Number of documents loaded:", len(documents))
print("\nSample content from the first document:\n")
print(documents[0].page_content[:400])


C:\Users\abhis\AppData\Local\Temp\ipykernel_8712\1203109683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Number of documents loaded: 2

Sample content from the first document:

Personal Knowledge Assistant - Internal Notes

What is a Personal Knowledge Assistant?
A Personal Knowledge Assistant is the small project I've been building across these
GenAI assignments. The idea is simple: instead of a new employee searching through five
different HR and IT documents to find one answer, they just ask the assistant a question
in plain English and it pulls the answer from the co


### Task 2: Text Splitting

Same splitter I've used in every RAG assignment so far - just calling out the
`chunk_size` and `chunk_overlap` explicitly since that's what this task is
actually asking about.


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)

print("chunk_size:", 500, " chunk_overlap:", 100)
print("Documents split into", len(chunks), "chunks")


chunk_size: 500  chunk_overlap: 100
Documents split into 18 chunks


## PART 2 — Vector Store & Retriever

### Task 3: Create Embeddings

Going with Hugging Face here since it runs locally without needing any API
credits - same choice as Assignment 25/26.


In [4]:
embeddings = None
try:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    test_vec = embeddings.embed_query("test")
    print("Embedding model ready. Vector length:", len(test_vec))
except Exception as e:
    print("Couldn't load the embedding model:", e)
    print("(Needs internet access the first time, to download the model.)")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model ready. Vector length: 384


### Task 4: Store Embeddings in Vector Store

FAISS again, same as every RAG assignment so far - it's local, no server to
run, and I already know the pattern.


In [5]:
retriever = None
faiss_db = None

if embeddings is not None:
    try:
        from langchain_community.vectorstores import FAISS
        faiss_db = FAISS.from_documents(chunks, embeddings)
        retriever = faiss_db.as_retriever(search_kwargs={"k": 3})
        print("Vector store built. Quick test:")
        for doc in retriever.invoke("What is the leave policy?"):
            print("-", doc.page_content[:100].replace("\n", " "))
    except Exception as e:
        print("Couldn't build the vector store:", e)
else:
    print("Skipping - no embedding model available.")


Vector store built. Quick test:
- Leave Policy Employees get 18 paid leaves per calendar year, plus public holidays as per the state c
- Leave Policy - Full Details Full-time employees accrue 18 paid leaves per calendar year, credited at
- allowed but should be regularized on the portal within 3 working days of returning, along with a doc


## PART 3 — Prompt with Message History

### Task 5: RAG Prompt Template

Three pieces here: a system message with the RAG instructions (answer only
from context, say "I don't know" otherwise), a `MessagesPlaceholder` for
whatever's been said earlier in the conversation, and the current question
as a human message.


In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the Personal Knowledge Assistant for new employees. Answer the "
     "question using ONLY the context below - do not use outside knowledge. "
     "If the answer isn't in the context, say 'I don't know based on the "
     "documents I have.' instead of guessing.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

# quick check that it renders correctly with an empty history
rendered = rag_prompt.format_messages(context="(sample context)", chat_history=[], question="test question")
for msg in rendered:
    print(f"[{msg.type}] {msg.content[:80]}")


[system] You are the Personal Knowledge Assistant for new employees. Answer the question 
[human] test question


## PART 4 — Q&A RAG Chain with Message History

### Task 6: Build RAG Chain

Question goes to the retriever, the retrieved chunks get formatted into
`context`, and both `context` and the running `chat_history` get fed into the
prompt before it reaches the LLM.


In [7]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

llm = None
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2", temperature=0.2)
    llm.invoke("say ok")
    print("Ollama is up, llama3.2 responded.")
except Exception as e:
    llm = None
    print("Couldn't reach Ollama:", e)
    print("The chain cells below will print a placeholder instead of a real answer.")


Ollama is up, llama3.2 responded.


In [8]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = None
if retriever is not None and llm is not None:
    context_step = RunnablePassthrough.assign(
        context=(lambda x: x["question"]) | retriever | RunnableLambda(format_docs)
    )
    rag_chain = context_step | rag_prompt | llm | StrOutputParser()
    print("RAG chain with message history is ready.")
else:
    print("Skipping - need both a retriever and Ollama available to build the real chain.")


RAG chain with message history is ready.


### Task 7: Maintain Message History

Just a plain Python list of `HumanMessage`/`AIMessage` objects that gets
appended to after every turn, and passed straight into the
`MessagesPlaceholder` on the next call.


In [9]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

def ask(question):
    if rag_chain is None:
        return "[No working chain right now - Ollama or the vector store isn't available]"
    try:
        answer = rag_chain.invoke({"question": question, "chat_history": chat_history})
        chat_history.append(HumanMessage(content=question))
        chat_history.append(AIMessage(content=answer))
        return answer
    except Exception as e:
        return f"[Question failed: {e}]"

print(ask("What is the leave policy?"))
print("\nMessages stored so far:", len(chat_history))


According to the provided context, the leave policy is as follows:

- Full-time employees accrue 18 paid leaves per calendar year, credited at the start of the year rather than month by month.
- Unused leaves up to a maximum of 8 days can be carried forward into the next calendar year; anything above that is forfeited.
- Leave requests must be submitted at least 2 days in advance through the HR portal, except in case of a medical emergency, where it can be informed after the fact along with a doctor's note.
- Sick leave taken without notice is allowed but should be regularized on the portal within 3 working days of returning, along with a doctor's note if the absence was longer than 2 days.
- Maternity and paternity leave follow separate statutory policies and are not deducted from the standard 18-day pool.

Messages stored so far: 2


### Task 8: Trimming Chat History

A long-running conversation would eventually blow past the model's context
window if I just kept appending forever, so I'm capping it - keep only the
most recent N messages, dropping the oldest ones once that limit is passed.
This part doesn't need the LLM at all, so I can actually test it directly
with some fake conversation turns.


In [10]:
from langchain_core.messages import trim_messages

def trim_history(history, max_messages=6):
    # keep only the most recent `max_messages` messages, oldest first out
    return trim_messages(
        history,
        max_tokens=max_messages,
        token_counter=len,       # counting messages, not real tokens, for simplicity
        strategy="last",
        start_on="human",
    )

# building a fake 8-message (4-turn) history to see the trimming actually work
fake_history = []
for i in range(1, 5):
    fake_history.append(HumanMessage(content=f"question number {i}"))
    fake_history.append(AIMessage(content=f"answer number {i}"))

print("Before trimming:", len(fake_history), "messages")
trimmed = trim_history(fake_history, max_messages=4)
print("After trimming to max_messages=4:", len(trimmed), "messages")
for m in trimmed:
    print(f"  [{m.type}] {m.content}")


Before trimming: 8 messages
After trimming to max_messages=4: 4 messages
  [human] question number 3
  [ai] answer number 3
  [human] question number 4
  [ai] answer number 4


In [11]:
# wiring the trim step into ask() so it actually gets used turn by turn,
# not just demonstrated once above
def ask(question, max_history_messages=6):
    global chat_history
    if rag_chain is None:
        return "[No working chain right now - Ollama or the vector store isn't available]"
    try:
        answer = rag_chain.invoke({"question": question, "chat_history": chat_history})
        chat_history.append(HumanMessage(content=question))
        chat_history.append(AIMessage(content=answer))
        chat_history = trim_history(chat_history, max_messages=max_history_messages)
        return answer
    except Exception as e:
        return f"[Question failed: {e}]"

print("ask() now trims chat_history down to the last", 6, "messages after every turn.")


ask() now trims chat_history down to the last 6 messages after every turn.


## PART 5 — Testing the Conversational RAG Bot

### Task 9: Multi-Turn Q&A Testing

Resetting the history first so this test starts clean, then running an
initial factual question, a follow-up that only makes sense with the
previous answer in mind, and a short clarification question.


In [12]:
chat_history = []

turns = [
    "What is the leave policy?",                       # initial factual question
    "What about carrying leaves over to next year?",     # follow-up referencing the previous answer
    "Can you explain that more simply?",                  # clarification question
]

for q in turns:
    print("-" * 60)
    print("You:", q)
    print("Bot:", ask(q))

print("\nMessages in history after this test:", len(chat_history))


------------------------------------------------------------
You: What is the leave policy?
Bot: The leave policy allows full-time employees to accrue 18 paid leaves per calendar year, credited at the start of the year.
------------------------------------------------------------
You: What about carrying leaves over to next year?
Bot: Unused leaves up to a maximum of 8 days can be carried forward into the next calendar year, but anything above that is forfeited.
------------------------------------------------------------
You: Can you explain that more simply?
Bot: I don't know based on the documents I have.

Messages in history after this test: 6


Couldn't actually verify grounded, context-preserving answers here since
neither Ollama nor the embedding model came through in this environment - but
the shape of the test is right: if this were working, I'd want the second
answer to clearly build on the first (carry-forward detail) rather than
re-explaining the whole leave policy from scratch, and the third answer to
simplify the *same* information instead of introducing something new.


## PART 6 — Mini Project: Conversational RAG Assistant

### Task 10: Build Final Chatbot

Putting Tasks 6-8 together into one small interactive loop - the actual
"chatbot" instead of just calling `ask()` in separate cells. A Streamlit UI
was optional here; I kept it to a plain input loop since the restrictions
said to focus on grounding and conversational flow, not the interface.


In [13]:
def run_chatbot():
    global chat_history
    chat_history = []
    print("Personal Knowledge Assistant - type 'exit' to quit\n")

    while True:
        user_input = input("You: ")
        if user_input.strip().lower() == "exit":
            print("Bot: Catch you later!")
            break

        reply = ask(user_input)
        print("Bot:", reply)
        print()

# not calling run_chatbot() automatically in this notebook since it blocks
# on input() - uncomment the line below to actually chat with it interactively
# run_chatbot()


## Task 11: Observations & Insights

**1. Difference between normal RAG and conversational RAG**
Normal RAG treats every question as a clean slate - it retrieves and answers
based only on that one question, with no idea what was asked before.
Conversational RAG adds a `chat_history` on top of that, so a follow-up like
"what about the previous point?" actually has something to point back to.
The retrieval step itself doesn't really change - it's the prompt (via
`MessagesPlaceholder`) that now carries the earlier turns along with the
retrieved context.

**2. Role of message history in follow-up questions**
Without it, a question like "can you explain that more simply?" is
meaningless to the model - there's no "that" to refer to. The message history
is what gives pronouns and vague references something concrete to resolve
against, so the model can actually connect "that" back to whatever it said
two turns ago.

**3. Trade-offs between long memory and performance**
Keeping the entire conversation forever means every single call sends more
and more text to the model - slower responses, higher cost (if using a paid
API), and eventually it just won't fit in the context window at all. Trimming
keeps things fast and cheap, but it comes at the cost of the bot genuinely
forgetting anything that happened before the cutoff - if someone references
something from message 1 in a 50-message conversation, a trimmed history just
won't have it anymore.

**4. How trimming affects answer quality**
As long as the trimmed-away messages weren't relevant to the current
question, there's basically no downside - the model still has the retrieved
document context plus enough recent conversation to follow along. The risk
is purely for long-range references: if someone circles back to something
from much earlier in a long conversation, a bot with trimmed history will
have no way to recall it and will either say it doesn't know or misunderstand
the follow-up entirely.


## Final note

The retrieval half of this assignment is really nothing new - same
load/split/embed/store pipeline as every RAG assignment before this one. The
actual new piece is the `chat_history` list plus `MessagesPlaceholder`, and
seeing that trimming is just list slicing dressed up with LangChain's message
types - `trim_messages` isn't doing anything I couldn't have written by hand,
it's just a tidier, more standard way to do it.
